In [30]:
import pandas as pd

# Load train and test datasets
df1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
df2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
df3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
test_df1 = pd.read_csv("dataset/test/test_source1.tsv", sep="\t")
test_df2 = pd.read_csv("dataset/test/test_source2.tsv", sep="\t")
test_df3 = pd.read_csv("dataset/test/test_source3.tsv", sep="\t")


In [31]:
# Check missing values
print("Source 1 nulls:\n", df1.isnull().sum())
print("\nSource 2 nulls:\n", df2.isnull().sum())
print("\nSource 3 nulls:\n", df3.isnull().sum())


Source 1 nulls:
 entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Source 2 nulls:
 entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

Source 3 nulls:
 entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64


In [32]:
import re
import unicodedata
import pandas as pd
from anyascii import anyascii

# ============================================================
# 1. REGEX PATTERNS & NORMALIZATION LOOKUPS
# ============================================================

# Common canonical mappings for legal suffixes
# (Multi-word specific patterns matched before single-word substrings)
LEGAL_SUFFIX_MAP = [
    (re.compile(r"\b(private\s+limited|pvt\s+ltd|private\s+ltd|pvt\s+limited)\b", re.IGNORECASE), "pvt ltd"),
    (re.compile(r"\b(corporation|incorporated|inc|corp)\b", re.IGNORECASE), "corp"),
    (re.compile(r"\b(limited\s+liability\s+company|llc)\b", re.IGNORECASE), "llc"),
    (re.compile(r"\b(limited\s+liability\s+partnership|llp)\b", re.IGNORECASE), "llp"),
    (re.compile(r"\b(public\s+limited\s+company|plc)\b", re.IGNORECASE), "plc"),
    (re.compile(r"\b(limited|ltd)\b", re.IGNORECASE), "ltd"),
    (re.compile(r"\b(sasu|sas)\b", re.IGNORECASE), "sas"),
    (re.compile(r"\b(sarl)\b", re.IGNORECASE), "sarl"),
    (re.compile(r"\b(eurl)\b", re.IGNORECASE), "eurl"),
    (re.compile(r"\b(sci)\b", re.IGNORECASE), "sci"),
    (re.compile(r"\b(snc)\b", re.IGNORECASE), "snc"),
]

URL_PREFIX_REGEX = re.compile(r"https?://|www\.", re.IGNORECASE)
DOMAIN_REGEX = re.compile(r"\.(com|org|net|in|fr|co|io|biz|info|gov|us)\b", re.IGNORECASE)

STOPWORDS = {"of", "the", "and", "at", "for", "in", "a", "an", "by", "de", "la", "le", "les", "du", "des"}

COUNTRY_MAP = {
    "us": "us", "usa": "us", "united states": "us",
    "india": "in", "ind": "in",
    "france": "fr", "fr": "fr"
}

ADDRESS_ABBREVIATIONS = {
    # US / UK / India
    r"\brd\.?\b": "road",
    r"\bst\.?\b": "street",
    r"\bave?\.?\b": "avenue",
    r"\bdr\.?\b": "drive",
    r"\bblvd\.?\b": "boulevard",
    r"\bln\.?\b": "lane",
    r"\bct\.?\b": "court",
    r"\bhwy\.?\b": "highway",
    r"\bpkwy\.?\b": "parkway",
    # Unit / apartment
    r"\bapt\.?\b": "unit",
    r"\bapartment\b": "unit",
    r"\bste\.?\b": "unit",
    r"\bsuite\b": "unit",
    r"\bpmb\b": "unit",
    # France
    r"\br\.(?=\s|$)": "rue",
    r"\bbd\.?\b": "boulevard",
    r"\bbvd\b": "boulevard",
    r"\ball\.?\b": "allee",
    r"\bimp\.?\b": "impasse",
    r"\bpl\.?\b": "place",
    r"\bche?\.?\b": "chemin",
    r"\brte\b": "route"
}

ORDINALS = {
    r"\b1st\b": "1", r"\bfirst\b": "1",
    r"\b2nd\b": "2", r"\bsecond\b": "2",
    r"\b3rd\b": "3", r"\bthird\b": "3",
    r"\b4th\b": "4", r"\bfourth\b": "4",
    r"\b5th\b": "5", r"\bfifth\b": "5"
}

US_STATES = {
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas",
    "ca": "california", "co": "colorado", "ct": "connecticut", "de": "delaware",
    "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas",
    "ky": "kentucky", "la": "louisiana", "me": "maine", "md": "maryland",
    "ma": "massachusetts", "mi": "michigan", "mn": "minnesota", "ms": "mississippi",
    "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma",
    "or": "oregon", "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina",
    "sd": "south dakota", "tn": "tennessee", "tx": "texas", "ut": "utah",
    "vt": "vermont", "va": "virginia", "wa": "washington", "wv": "west virginia",
    "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia"
}

US_STATE_REGEX = re.compile(r"\b(" + "|".join(sorted(US_STATES, key=len, reverse=True)) + r")\b")


In [33]:
# ============================================================
# 2. FIELD-LEVEL CLEANING FUNCTIONS
# ============================================================

def clean_business_name(text: str) -> str:
    """
    Transliterates non-Latin scripts to ASCII, lowercases, removes URLs/domains,
    standardizes legal suffixes to common canonical values (e.g. private limited -> pvt ltd,
    corporation/inc -> corp), cleans punctuation, and removes stopwords.
    """
    if not isinstance(text, str) or not text.strip() or text.strip().lower() == "nan":
        return ""

    # 1. Unicode normalization and transliteration to ASCII
    s = unicodedata.normalize("NFKC", text)
    s = anyascii(s).lower()

    # 2. Remove URL prefixes and domain extensions
    s = URL_PREFIX_REGEX.sub("", s)
    s = DOMAIN_REGEX.sub(" ", s)

    # 3. Clean punctuation (handles abbreviations like pvt. ltd. -> pvt ltd)
    s = re.sub(r"[^\w\s]", " ", s)

    # 4. Standardize legal / corporate suffixes to common canonical forms
    for pattern, repl in LEGAL_SUFFIX_MAP:
        s = pattern.sub(repl, s)

    # 5. Remove stopwords and normalize whitespace
    tokens = [w for w in s.split() if w not in STOPWORDS]
    return " ".join(tokens)


def clean_address_text(addr: str, country_code: str = "") -> str:
    """
    Transliterates address to ASCII, normalizes street abbreviations,
    expands US state abbreviations (if country is US), standardizes ordinals,
    and removes punctuation while PRESERVING postal/PIN code digits.
    """
    if not isinstance(addr, str) or not addr.strip() or addr.strip().lower() == "nan":
        return ""

    # 1. Unicode normalization and transliteration to ASCII
    s = unicodedata.normalize("NFKC", addr)
    s = anyascii(s).lower()

    # 2. Standardize street abbreviations
    for pattern, repl in ADDRESS_ABBREVIATIONS.items():
        s = re.sub(pattern, repl, s)

    # 3. Standardize ordinals (1st -> 1, 2nd -> 2, etc.)
    for pattern, repl in ORDINALS.items():
        s = re.sub(pattern, repl, s)

    # 4. Expand US states ONLY for US addresses (avoids Indian address collision where 'in' -> 'indiana')
    if country_code in ("us", "usa"):
        s = US_STATE_REGEX.sub(lambda m: US_STATES[m.group(1)], s)

    # 5. Remove punctuation but KEEP all alphanumeric characters (postal codes / PIN codes / digits preserved)
    s = re.sub(r"[^\w\s]", " ", s)

    # 6. Normalize unit noise
    s = re.sub(r"\bunit\s+\w+\b", " ", s)

    # 7. Normalize whitespace
    return re.sub(r"\s+", " ", s).strip()


In [34]:
# ============================================================
# 3. DATASET PROCESSOR
# ============================================================

def process_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalizes country, business_name, and business_address in-place.
    Directly generates only the desired normalized columns:
    - country_clean
    - clean_name (with standardized corporate suffixes like pvt ltd, corp)
    - clean_address (with preserved postal/PIN code digits)
    No intermediate or unwanted columns are created, so no drop is needed.
    """
    
    # 1. Clean country code
    country_values = df["country"].fillna("").astype(str).str.lower().str.strip()
    df["country_clean"] = country_values.map(COUNTRY_MAP).fillna(country_values)

    # 2. Clean business name (standardizes legal suffixes instead of removing)
    df["clean_name"] = df["business_name"].fillna("").apply(clean_business_name)

    # 3. Clean business address (preserves postal code numbers inside clean_address)
    df["clean_address"] = [
        clean_address_text(addr, c)
        for addr, c in zip(df["business_address"], df["country_clean"])
    ]

    return df


In [35]:
# ============================================================
# 4. APPLY NORMALIZATION TO TRAIN AND TEST DATASETS
# ============================================================

for data in (df1, df2, df3, test_df1, test_df2, test_df3):
    process_dataset(data)

print("Train and test normalization complete!")


Train and test normalization complete!


In [37]:
# Verify final schema and cleaned records on df1
df1.head(10)


,entity_id,business_name,business_address,country,country_clean,clean_name,clean_address
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,us,orelee s barbershop,1795 westchester drive high point north carolina
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,us,prime money,17560 ellis road tahlequah oklahoma
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,us,b retail corp,1712 montebello avenue phoenix arizona
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,us,christ chapel,2100 cameron drive g dundalk maryland
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,in,prabhav business center,797 lake town block a kolkata howrah west bengal
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,us,custom wealth services llc,ohio columbus 5559 orville avenue
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,in,consulting nyasa nursing pvt ltd,2505 tower 1 oakwood runwal greens mulund gore...
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,us,nexus anchor rain,1111 church street nashville tennessee
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,us,moore bitwise corp,337 oakland avenue michigan city indiana
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,us,dermatology green medicine,294 meadowcreek drive 2 village of pewaukee wi...
